In [3]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, accuracy_score
from sklearn.ensemble import VotingClassifier
from sklearn.model_selection import cross_val_score
from sklearn.feature_selection import mutual_info_classif
from sklearn.feature_selection import SelectKBest

In [6]:
unsw_train = pd.read_csv("../data/UNSW_NB15_training-set.csv",encoding='utf-8')
unsw_test = pd.read_csv("../data/UNSW_NB15_testing-set.csv",encoding='utf-8')
unsw = pd.concat([unsw_train, unsw_test]).sample(frac=1)

In [8]:
unsw.columns

Index(['id', 'dur', 'proto', 'service', 'state', 'spkts', 'dpkts', 'sbytes',
       'dbytes', 'rate', 'sttl', 'dttl', 'sload', 'dload', 'sloss', 'dloss',
       'sinpkt', 'dinpkt', 'sjit', 'djit', 'swin', 'stcpb', 'dtcpb', 'dwin',
       'tcprtt', 'synack', 'ackdat', 'smean', 'dmean', 'trans_depth',
       'response_body_len', 'ct_srv_src', 'ct_state_ttl', 'ct_dst_ltm',
       'ct_src_dport_ltm', 'ct_dst_sport_ltm', 'ct_dst_src_ltm',
       'is_ftp_login', 'ct_ftp_cmd', 'ct_flw_http_mthd', 'ct_src_ltm',
       'ct_srv_dst', 'is_sm_ips_ports', 'attack_cat', 'label'],
      dtype='str')

In [19]:
unsw.drop(columns='attack_cat', inplace=True)

In [21]:
# attack_n = []
# for i in nsl.attack :
#   if i == 'normal':
#     attack_n.append(0)
#   else:
#     attack_n.append(1)
# nsl['attack'] = attack_n

In [22]:
unsw_obj=unsw.select_dtypes(['object']).columns

C:\Users\LENOVO\AppData\Local\Temp\ipykernel_19332\3474970430.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  unsw_obj=unsw.select_dtypes(['object']).columns


In [24]:
le=LabelEncoder()

for i in unsw_obj:
  unsw[i]=le.fit_transform(unsw[i])

In [25]:
def mutual_info_select(dataset):
    x=dataset.drop(['label'], axis=1)
    y=dataset['label'].copy()
    x_train, x_test, y_train, y_test = train_test_split(x,y , test_size=0.3, random_state=40)
    mutual_info = mutual_info_classif(x_train, y_train)
    mutual_info = pd.Series(mutual_info)
    mutual_info.index = x_train.columns
    mutual_info.sort_values(ascending=False)
    sel_five_cols = SelectKBest(mutual_info_classif, k=10)
    sel_five_cols.fit(x_train, y_train)
    nsl_col=x_train.columns[sel_five_cols.get_support()]
    return nsl_col

In [26]:
# EFPA implementation
def EFPA(dataset):
  dataset=dataset.drop(['label'], axis=1)
  gamma = 0.01
  lam = np.abs(np.random.uniform(0.30, 1.99))
  population_size = 200
  switch_probability = 0.8
  def levy_flight(size, lam):
      return np.random.standard_cauchy(size) / (np.random.uniform(0, 1, size) ** (1 / lam))

  def select_features(dataset, pollen, threshold=0.5):
      return dataset.columns[pollen > threshold]

  def fitness_function(pollen):
      return np.sum(pollen)

  pollen_elements = np.random.rand(population_size, len(dataset.columns))

  fitness_values = np.array([fitness_function(pollen) for pollen in pollen_elements])
  best_pollen_index = np.argmax(fitness_values)
  abest = pollen_elements[best_pollen_index]


  for iteration in range(0, 200):

      levy_flight_values = levy_flight((population_size, len(dataset.columns)), lam)

      new_pollen_elements = pollen_elements + levy_flight_values * (abest - pollen_elements)

      mutation_scaling_factor = np.random.uniform(0, 1, size=(population_size, 1))
      new_pollen_elements += gamma * mutation_scaling_factor * (new_pollen_elements - np.roll(new_pollen_elements, shift=1, axis=0))

      new_fitness_values = np.array([fitness_function(pollen) for pollen in new_pollen_elements])

      update_mask = new_fitness_values > fitness_values
      pollen_elements[update_mask] = new_pollen_elements[update_mask]
      fitness_values[update_mask] = new_fitness_values[update_mask]

      best_solution_index = np.argmax(fitness_values)
      abest = pollen_elements[best_solution_index]

  best_solution_index = np.argmax(fitness_values)
  abest = pollen_elements[best_solution_index]

  selected_features = select_features(dataset, abest)
  return(selected_features)

In [27]:
def correlation(dataset):
    dataset=dataset.drop(['label'], axis=1)
    corr_matrix = dataset.corr().abs()
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    to_drop = [column for column in upper.columns if any(upper[column] > 0.95)]
    correlation_nsl=np.setdiff1d(np.array(unsw.columns),to_drop)
    return correlation_nsl


In [28]:
def train_test(col):

    x=unsw[col]
    y=unsw['label'].copy()

    x_train, x_test, y_train, y_test = train_test_split(x,y , test_size=0.3, random_state=40)

    scalar=StandardScaler()
    x_train=scalar.fit_transform(x_train)
    x_test = scalar.fit_transform(x_test)


    # Train logistic regression
    logistic_reg = LogisticRegression().fit(x_train, y_train)
    y_pred_logistic = logistic_reg.predict(x_test)
    logistic_reg_clss_rep = [logistic_reg.score(x_train, y_train), logistic_reg.score(x_test, y_test), str(np.round(accuracy_score(y_test, y_pred_logistic), 3)), classification_report(y_test, y_pred_logistic)]

    # Train decision tree
    decision_tree = DecisionTreeClassifier().fit(x_train, y_train)
    y_pred_dt = decision_tree.predict(x_test)
    decision_tree_clss_rep = [decision_tree.score(x_train, y_train), decision_tree.score(x_test, y_test), str(np.round(accuracy_score(y_test, y_pred_dt), 3)), classification_report(y_test, y_pred_dt)]

    # Train random forest
    random_forest = RandomForestClassifier().fit(x_train, y_train)
    y_pred_rf = random_forest.predict(x_test)
    random_forest_clss_rep = [random_forest.score(x_train, y_train), random_forest.score(x_test, y_test), str(np.round(accuracy_score(y_test, y_pred_rf), 3)), classification_report(y_test, y_pred_rf)]

    # Train KNN
    knn = KNeighborsClassifier().fit(x_train, y_train)
    y_pred_knn = knn.predict(x_test)
    knn_clss_rep = [knn.score(x_train, y_train), knn.score(x_test, y_test), str(np.round(accuracy_score(y_test, y_pred_knn), 3)), classification_report(y_test, y_pred_knn)]

#     # Define the ensemble
#     ensemble = VotingClassifier(estimators=[
#         ('lr', logistic_reg),
#         ('dt', decision_tree),
#         ('rf', random_forest),
#         ('knn', knn)
#     ], voting='hard').fit(x_train, y_train)
#     y_pred_ensemble = ensemble.predict(x_test)
#     ensemble_clss_rep = [ensemble.score(x_train, y_train),ensemble.score(x_test, y_test),str(np.round(accuracy_score(y_test, y_pred_ensemble), 3)),classification_report(y_test, y_pred_ensemble)]
    
#     return [logistic_reg_clss_rep,decision_tree_clss_rep,random_forest_clss_rep,knn_clss_rep,ensemble_clss_rep]
    return [logistic_reg_clss_rep,decision_tree_clss_rep,random_forest_clss_rep,knn_clss_rep]

In [29]:
# def train_test(col):
#     x = unsw[col]
#     y = unsw['label'].copy()

#     x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.3, random_state=40)

#     scalar = StandardScaler()
#     x_train = scalar.fit_transform(x_train)
#     x_test = scalar.fit_transform(x_test)

#     # Cross-validation for logistic regression
#     logistic_reg_cv_scores = cross_val_score(LogisticRegression(), x_train, y_train, cv=5)
#     logistic_reg_cv_mean_score = np.mean(logistic_reg_cv_scores)
#     logistic_reg = LogisticRegression().fit(x_train, y_train)
#     y_pred_logistic = logistic_reg.predict(x_test)
#     logistic_reg_clss_rep = [logistic_reg.score(x_train, y_train), logistic_reg.score(x_test, y_test), str(np.round(accuracy_score(y_test, y_pred_logistic), 3)), classification_report(y_test, y_pred_logistic)]

#     # Cross-validation for decision tree
#     decision_tree_cv_scores = cross_val_score(DecisionTreeClassifier(), x_train, y_train, cv=5)
#     decision_tree_cv_mean_score = np.mean(decision_tree_cv_scores)
#     decision_tree = DecisionTreeClassifier().fit(x_train, y_train)
#     y_pred_dt = decision_tree.predict(x_test)
#     decision_tree_clss_rep = [decision_tree.score(x_train, y_train), decision_tree.score(x_test, y_test), str(np.round(accuracy_score(y_test, y_pred_dt), 3)), classification_report(y_test, y_pred_dt)]

#     # Cross-validation for random forest
#     random_forest_cv_scores = cross_val_score(RandomForestClassifier(), x_train, y_train, cv=5)
#     random_forest_cv_mean_score = np.mean(random_forest_cv_scores)
#     random_forest = RandomForestClassifier().fit(x_train, y_train)
#     y_pred_rf = random_forest.predict(x_test)
#     random_forest_clss_rep = [random_forest.score(x_train, y_train), random_forest.score(x_test, y_test), str(np.round(accuracy_score(y_test, y_pred_rf), 3)), classification_report(y_test, y_pred_rf)]

#     # Cross-validation for KNN
#     knn_cv_scores = cross_val_score(KNeighborsClassifier(), x_train, y_train, cv=5)
#     knn_cv_mean_score = np.mean(knn_cv_scores)
#     knn = KNeighborsClassifier().fit(x_train, y_train)
#     y_pred_knn = knn.predict(x_test)
#     knn_clss_rep = [knn.score(x_train, y_train), knn.score(x_test, y_test), str(np.round(accuracy_score(y_test, y_pred_knn), 3)), classification_report(y_test, y_pred_knn)]

#     return [logistic_reg_clss_rep,decision_tree_clss_rep,random_forest_clss_rep,knn_clss_rep]


In [30]:
normal=train_test(EFPA(unsw))
print('Logistic regression Training accuracy = ',normal[0][0])
print('Logistic regression Testing accuracy = ',normal[0][1])
print("------------------------------------------------")
print("Logistic regression  accuracy  : " + normal[0][2])
print("------------------------------------------------")
print("Logistic regression Classification report : ",normal[0][3])

print("~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~")

print('Decision Tree Training accuracy = ',normal[1][0])
print('Decision Tree Testing accuracy = ',normal[1][1])
print("------------------------------------------------")
print("Decision Tree  accuracy  : " + normal[1][2])
print("------------------------------------------------")
print("Decision Tree Classification report : ",normal[1][3])

print("~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~")

print('Random Forest Training accuracy = ',normal[2][0])
print('Random Forest Testing accuracy = ',normal[2][1])
print("------------------------------------------------")
print("Random Forest  accuracy  : " + normal[2][2])
print("------------------------------------------------")
print("Random Forest Classification report : ",normal[2][3])

print("~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~")

print('KNN Training accuracy = ',normal[3][0])
print('KNN Testing accuracy = ',normal[3][1])
print("------------------------------------------------")
print("KNN  accuracy  : " + normal[3][2])
print("------------------------------------------------")
print("KNN Classification report : ",normal[3][3])

# print("~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~")

# print('Ensemble Training accuracy = ',normal[4][0])
# print('Ensemble Testing accuracy = ',normal[4][1])
# print("------------------------------------------------")
# print("Ensemble  accuracy  : " + normal[4][2])
# print("------------------------------------------------")
# print("Ensemble Classification report : ",normal[4][3])




C:\Users\LENOVO\AppData\Local\Temp\ipykernel_19332\1081673924.py:28: RuntimeWarning: overflow encountered in multiply
  new_pollen_elements = pollen_elements + levy_flight_values * (abest - pollen_elements)
C:\Users\LENOVO\AppData\Local\Temp\ipykernel_19332\1081673924.py:28: RuntimeWarning: invalid value encountered in subtract
  new_pollen_elements = pollen_elements + levy_flight_values * (abest - pollen_elements)
C:\Users\LENOVO\AppData\Local\Temp\ipykernel_19332\1081673924.py:31: RuntimeWarning: invalid value encountered in subtract
  new_pollen_elements += gamma * mutation_scaling_factor * (new_pollen_elements - np.roll(new_pollen_elements, shift=1, axis=0))


Logistic regression Training accuracy =  0.8765433467686047
Logistic regression Testing accuracy =  0.875928177796176
------------------------------------------------
Logistic regression  accuracy  : 0.876
------------------------------------------------
Logistic regression Classification report :                precision    recall  f1-score   support

           0       0.88      0.76      0.82     27973
           1       0.87      0.94      0.91     49329

    accuracy                           0.88     77302
   macro avg       0.88      0.85      0.86     77302
weighted avg       0.88      0.88      0.87     77302

~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
Decision Tree Training accuracy =  0.9999944558715093
Decision Tree Testing accuracy =  0.8668598483868464
------------------------------------------------
Decision Tree  accuracy  : 0.867
------------------------------------------------
Decision Tree Classification report :                precision    recall  f1-score   s

In [31]:
from xgboost import XGBClassifier

x=unsw[EFPA(unsw)]
y=unsw['label'].copy()

# Split the data into training and testing sets
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.3, random_state=42)

scalar=StandardScaler()
x_train=scalar.fit_transform(x_train)
x_test = scalar.fit_transform(x_test)

# Initialize the XGBoost classifier
xgb_classifier = XGBClassifier()

# Train the classifier
xgb_classifier.fit(x_train, y_train)

# Make predictions on the test set
y_pred = xgb_classifier.predict(x_test)

# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

# Print classification report
print("Classification Report:")
print(classification_report(y_test, y_pred))

C:\Users\LENOVO\AppData\Local\Temp\ipykernel_19332\1081673924.py:28: RuntimeWarning: overflow encountered in multiply
  new_pollen_elements = pollen_elements + levy_flight_values * (abest - pollen_elements)
C:\Users\LENOVO\AppData\Local\Temp\ipykernel_19332\1081673924.py:28: RuntimeWarning: invalid value encountered in subtract
  new_pollen_elements = pollen_elements + levy_flight_values * (abest - pollen_elements)
C:\Users\LENOVO\AppData\Local\Temp\ipykernel_19332\1081673924.py:31: RuntimeWarning: invalid value encountered in subtract
  new_pollen_elements += gamma * mutation_scaling_factor * (new_pollen_elements - np.roll(new_pollen_elements, shift=1, axis=0))


Accuracy: 0.6330625339577243
Classification Report:
              precision    recall  f1-score   support

           0       0.49      0.89      0.64     27811
           1       0.89      0.49      0.63     49491

    accuracy                           0.63     77302
   macro avg       0.69      0.69      0.63     77302
weighted avg       0.75      0.63      0.63     77302

